# 문제 19. 이탈 기준선 탐색과 민감도

"90일 무구매 = 이탈"이라는 통념의 근거를 재구매 간격 데이터로 직접 검증합니다.
외부 모듈 없이, orders.csv 하나만으로 이 노트북 안에서 끝까지 풀어냅니다.

**전제**: `data/orders.csv`와 같은 위치에서 실행합니다.

## 0. 환경 설정

In [12]:
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 관측 기준일: 데이터 기간(2024 상반기) 종료 다음 날
REF_DATE = pd.Timestamp("2024-07-01")


## 1. 유효 주문 적재 및 정제

이 문제는 order_items 없이 **orders(주문 단위)만** 사용합니다.
1장에서 확정한 기준과 동일하게: 중복 제거 -> 날짜 파싱(coerce) -> 2024 상반기 필터 -> canceled/returned 제외.

In [ ]:
# orders : 주문 데이터 로드 및 전처리
# period_mask : 2024년 상반기 주문만 추출
# valid_orders: 유효 주문만 추출("canceled", "returned" 제외)
orders = pd.read_csv("./data/orders.csv")
orders = orders.drop_duplicates()

orders["order_datetime"] = pd.to_datetime(orders["order_datetime"], errors="coerce")
period_mask = (orders["order_datetime"] >= "2024-01-01") & (orders["order_datetime"] < "2024-07-01")
orders = orders[period_mask].copy()

valid_orders = orders[~orders["status"].isin(["canceled", "returned"])].copy()

print(f"유효 주문 수: {len(valid_orders):,}건")
print(f"고유 고객 수: {valid_orders['customer_id'].nunique():,}명")
valid_orders[["order_id", "customer_id", "order_datetime", "status"]].head()


유효 주문 수: 168,312건
고유 고객 수: 5,793명


,order_id,customer_id,order_datetime,status
0,25463,1077,2024-06-25 00:17:41,delivered
1,171088,2998,2024-05-28 19:35:20,delivered
2,27437,4066,2024-02-14 17:49:14,delivered
3,98425,3243,2024-05-08 16:37:36,delivered
4,22325,5331,2024-04-17 20:08:22,delivered


---
## 2. 고객별 시간순 정렬 + 이전 주문 시각 당겨오기

`sort_values`로 고객별·시간순 정렬 후, `groupby().shift(1)`로 **바로 이전 주문의 시각**을
같은 행에 나란히 붙입니다. 이래야 두 시각을 빼서 '간격'을 계산할 수 있습니다.

In [18]:
# orders : 주문 데이터 로드 및 전처리
# period_mask : 2024년 상반기 주문만 추출
# valid_orders: 유효 주문만 추출("canceled", "returned" 제외)
# cust_orders : 고객별 주문일자 정렬 및 이전 주문일자 계산
cust_orders = valid_orders.sort_values(["customer_id", "order_datetime"])

cust_orders["prev_order_datetime"] = cust_orders.groupby("customer_id")["order_datetime"].shift(1)

cust_orders[["customer_id", "order_datetime", "prev_order_datetime"]].head(8)


,customer_id,order_datetime,prev_order_datetime
36493,1000,2024-01-30 08:13:14,NaT
9266,1000,2024-02-01 10:46:11,2024-01-30 08:13:14
114393,1000,2024-04-09 01:44:40,2024-02-01 10:46:11
135242,1000,2024-04-12 21:33:49,2024-04-09 01:44:40
2553,1000,2024-04-30 04:32:04,2024-04-12 21:33:49
20132,1000,2024-05-06 17:39:14,2024-04-30 04:32:04
102427,1000,2024-06-06 13:37:23,2024-05-06 17:39:14
195170,1000,2024-06-20 18:13:10,2024-06-06 13:37:23


## 3. 재구매 간격(gap_days) 계산

고객의 **첫 주문**은 `prev_order_datetime`이 없어(`shift`로 당겨올 이전 값이 없음) `NaT`가 되고,
그 결과 `gap_days`도 결측이 됩니다. 이 결측은 자연스럽게 '주문 1회 고객'을 걸러내는 역할을 합니다.

In [ ]:
# orders : 주문 데이터 로드 및 전처리
# period_mask : 2024년 상반기 주문만 추출
# valid_orders: 유효 주문만 추출("canceled", "returned" 제외)
# cust_orders : 고객별 주문일자 정렬 및 이전 주문일자 계산(현재주문일자와 이전주문일자 간격 계산) 
# gap_dist : 재구매 간격(현재주문일자와 이전주문일자 간격) 분포
print(f"정제전 주문 행: {cust_orders.shape[0]:,}건")
cust_orders["gap_days"] = (cust_orders["order_datetime"] - cust_orders["prev_order_datetime"]).dt.days

# 간격이 없는(=주문 1회이거나 첫 주문인) 행
n_single = cust_orders["gap_days"].isna().sum()
print(f"간격이 없는(=주문 1회이거나 첫 주문인) 행: {n_single:,}건")

gap_dist = cust_orders["gap_days"].dropna()
print(f"재구매 간격 표본 수: {len(gap_dist):,}건")
gap_dist.describe()


정제전 주문 행: 168,312건
간격이 없는(=주문 1회이거나 첫 주문인) 행: 5,793건
재구매 간격 표본 수: 162,519건


count   162,519.00
mean          4.35
std           8.86
min           0.00
25%           0.00
50%           1.00
75%           5.00
max         138.00
Name: gap_days, dtype: float64

### (참고) 주문 1회 고객 수 별도 확인

위의 결측 건수는 '간격 행'의 결측이고, '고객 수' 기준으로는 아래처럼 별도로 셉니다.

In [ ]:
# orders : 주문 데이터 로드 및 전처리
# period_mask : 2024년 상반기 주문만 추출
# valid_orders: 유효 주문만 추출("canceled", "returned" 제외)
# cust_orders : 고객별 주문일자 정렬 및 이전 주문일자 계산(현재주문일자와 이전주문일자 간격 계산) 
# gap_dist : 재구매 간격(현재주문일자와 이전주문일자 간격) 분포
orders_per_customer = cust_orders.groupby("customer_id").size()
n_single_customers = (orders_per_customer == 1).sum()

print(f"전체 고객 수: {len(orders_per_customer):,}명")
print(f"주문 1회 고객(재구매 간격 계산 불가): {n_single_customers:,}명")


전체 고객 수: 5,793명
주문 1회 고객(재구매 간격 계산 불가): 841명


---
## 4. 재구매 간격 분포에서 후보 기준선(P75, P90) 도출

In [ ]:
# orders : 주문 데이터 로드 및 전처리
# period_mask : 2024년 상반기 주문만 추출
# valid_orders: 유효 주문만 추출("canceled", "returned" 제외)
# cust_orders : 고객별 주문일자 정렬 및 이전 주문일자 계산(현재주문일자와 이전주문일자 간격 계산) 
# gap_dist : 재구매 간격(현재주문일자와 이전주문일자 간격) 분포
p75, p90 = gap_dist.quantile([0.75, 0.90])

print(f"재구매 간격 P75: {p75:.0f}일")
print(f"재구매 간격 P90: {p90:.0f}일")
print("\n[해석] 재구매 고객의 75%는 P75일 이내에 다시 구매했고, 90%는 P90일 이내에 다시 구매했습니다.")


p75: 5.0
p75: 5.0
p75: 5.0
재구매 간격 P75: 5일
재구매 간격 P90: 13일

[해석] 재구매 고객의 75%는 P75일 이내에 다시 구매했고, 90%는 P90일 이내에 다시 구매했습니다.


---
## 5. 고객별 recency(마지막 구매 후 경과일) 계산

**주의**: 시각(datetime) 그대로 빼면 같은 날 오후에 산 경우 소수점 이하 시간 차이로 계산이 꼬일 수 있습니다.
그래서 `.dt.normalize()`로 시각을 00:00:00으로 맞춘 뒤, **날짜 단위**로 빼야 안전합니다.

In [ ]:
# orders : 주문 데이터 로드 및 전처리
# period_mask : 2024년 상반기 주문만 추출
# valid_orders: 유효 주문만 추출("canceled", "returned" 제외)
# cust_orders : 고객별 주문일자 정렬 및 이전 주문일자 계산(현재주문일자와 이전주문일자 간격 계산) 
# gap_dist : 재구매 간격(현재주문일자와 이전주문일자 간격) 분포
# last_purchase : 고객별 마지막 주문일자
# last_purchase_date : 고객별 마지막 주문일자(날짜 단위로 통일) yyyy-mm-dd
# recency : 고객별 마지막 주문일자 기준으로 관측 기준일(2024-07-01)까지의 경과 일수
# count   5,793.00
# mean       20.32
# std        33.86
# min         1.00
# 25%         2.00
# 50%         8.00
# 75%        20.00
# max       182.00
last_purchase = valid_orders.groupby("customer_id")["order_datetime"].max()
last_purchase_date = last_purchase.dt.normalize()  # 날짜 단위로 통일 (시:분:초 제거)

recency = (REF_DATE.normalize() - last_purchase_date).dt.days

print(f"recency 계산된 고객 수: {len(recency):,}명")
recency.describe()


recency 계산된 고객 수: 5,793명


count   5,793.00
mean       20.32
std        33.86
min         1.00
25%         2.00
50%         8.00
75%        20.00
max       182.00
Name: order_datetime, dtype: float64

---
## 6. 기준선 후보별 이탈률 민감도 표

이탈률 = `recency > 기준선` 인 고객의 비율입니다.
60·90·120일 같은 통념상 후보와, 데이터에서 직접 뽑은 P75·P90을 나란히 비교합니다.

In [33]:
# orders : 주문 데이터 로드 및 전처리
# period_mask : 2024년 상반기 주문만 추출
# valid_orders: 유효 주문만 추출("canceled", "returned" 제외)
# cust_orders : 고객별 주문일자 정렬 및 이전 주문일자 계산(현재주문일자와 이전주문일자 간격 계산) 
# gap_dist : 재구매 간격(현재주문일자와 이전주문일자 간격) 분포
# last_purchase : 고객별 마지막 주문일자
# last_purchase_date : 고객별 마지막 주문일자(날짜 단위로 통일) yyyy-mm-dd
# recency : 고객별 마지막 주문일자 기준으로 관측 기준일(2024-07-01)까지의 경과 일수
# count   5,793.00
# mean       20.32
# std        33.86
# min         1.00
# 25%         2.00
# 50%         8.00
# 75%        20.00
# max       182.00
candidates = sorted(set([60, 90, 120, int(p75), int(p90)]))
print(f"이탈 기준선 후보: {candidates}일")
print(recency)

sensitivity = pd.Series({
    f"{c}일": (recency > c).mean() for c in candidates
})

print("[기준선 후보별 이탈률]")
sensitivity.apply(lambda x: f"{x:.1%}")


이탈 기준선 후보: [5, 13, 60, 90, 120]일
customer_id
1000       11
1001        8
1002        6
1003        2
1004        2
         ... 
989712     42
989772    173
989922      2
989977    169
989995     35
Name: order_datetime, Length: 5793, dtype: int64
[기준선 후보별 이탈률]


5일      58.1%
13일     33.8%
60일      9.4%
90일      6.3%
120일     3.8%
dtype: str

---
## 7. 권고 기준선과 근거

In [10]:
recommend = int(round(p75, -1))  # P75를 10일 단위로 반올림해 1차 경보선으로 제안

print(
    f"[재구매 간격 분포] P75={p75:.0f}일 / P90={p90:.0f}일 "
    f"(주문 1회 고객 {n_single_customers:,}명은 분포에서 제외)\n"
)
print("[기준선 후보별 이탈률]")
print(sensitivity.apply(lambda x: f'{x:.1%}').to_string())
print(
    f"\n[권고] 재구매 간격 P75가 {p75:.0f}일 안팎이라면, 통념인 '90일 무구매=이탈'은 "
    "이 쇼핑몰 고객군에는 느슨한 기준입니다.\n"
    f"P75를 반올림한 {recommend}일을 1차 경보선으로 쓰고, P90({p90:.0f}일)을 확정 이탈선으로 "
    "이원화할 것을 권고합니다."
)


[재구매 간격 분포] P75=5일 / P90=13일 (주문 1회 고객 841명은 분포에서 제외)

[기준선 후보별 이탈률]
5일      58.1%
13일     33.8%
60일      9.4%
90일      6.3%
120일     3.8%

[권고] 재구매 간격 P75가 5일 안팎이라면, 통념인 '90일 무구매=이탈'은 이 쇼핑몰 고객군에는 느슨한 기준입니다.
P75를 반올림한 0일을 1차 경보선으로 쓰고, P90(13일)을 확정 이탈선으로 이원화할 것을 권고합니다.


## 8. 검열(censoring) — 관측 종료 근처 고객의 이탈 미확정

관측 종료일(6/30) 근처에 마지막 구매가 있는 고객은, recency가 아직 작아서 '이탈 아님'으로 보이지만
사실은 **관측이 끝나서 그 이후를 알 수 없는 상태**일 뿐입니다. 이런 고객이 정말 이탈했는지는
미래 데이터가 있어야 확정할 수 있으며, 이 개념을 **검열(censoring)**이라 부릅니다.

즉 지금 계산한 이탈률은 '6/30 시점까지 확인된 이탈률'이지, 모든 고객의 최종 이탈 여부를
확정한 값이 아니라는 점을 보고서에 명시해야 합니다.

In [11]:
near_cutoff = (REF_DATE.normalize() - last_purchase_date).dt.days
n_near_cutoff = (near_cutoff <= 30).sum()  # 예: 최근 30일 이내 구매한 고객 수

print(f"관측 종료일(6/30) 기준 최근 30일 이내 구매 고객: {n_near_cutoff:,}명")
print("-> 이들은 recency가 낮아 '이탈 아님'으로 보이지만, 7월 이후 데이터가 없어 실제 이탈 여부는 검열된 상태입니다.")


관측 종료일(6/30) 기준 최근 30일 이내 구매 고객: 4,834명
-> 이들은 recency가 낮아 '이탈 아님'으로 보이지만, 7월 이후 데이터가 없어 실제 이탈 여부는 검열된 상태입니다.
